# **ColabProSST**

ColabProSST brings the ProSST backbone into the SaprotHub workflow with a compact SaProt-style entry point. ProSST is not SaProt: it uses amino-acid `input_ids` together with official ProSST `ss_input_ids`, so every task needs either `structure_tokens`, `pdb_path`/`structure_path`, or structure tokens generated in this notebook.

Supported in this first version: structure-token conversion, zero-shot mutation effect prediction, protein-level classification fine-tuning, protein-level regression fine-tuning, checkpoint prediction, and optional Hugging Face upload.


In [ ]:
#@title **Click the run-button to prepare ColabProSST**
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

SAPROTHUB_REPO = 'https://github.com/Hello-github-code/SaprotHub.git' #@param {type:'string'}
SAPROTHUB_BRANCH = 'prosst' #@param {type:'string'}
DOWNLOAD_CSV_TEMPLATES = False #@param {type:'boolean'}

ROOT = Path(os.getcwd())
SAPROT_HOME = ROOT / 'SaprotHub'
PROSST_HOME = ROOT / 'ProSST'
PYG_HOME = ROOT / '.cache' / 'pyg'
HF_HOME = ROOT / '.cache' / 'huggingface'
SAPROT_REQUIRED = [
    Path('pyproject.toml'),
    Path('saprot/__init__.py'),
    Path('saprot/utils/colab_prosst_workflow.py'),
]
PROSST_REQUIRED = [
    Path('prosst/structure/get_sst_seq.py'),
    Path('prosst/structure/static/AE.pt'),
    Path('prosst/structure/static/2048.joblib'),
]

for path in [PYG_HOME, HF_HOME, ROOT / 'prosst_uploads', ROOT / 'prosst_structure_assets']:
    path.mkdir(parents=True, exist_ok=True)
os.environ['PYG_HOME'] = str(PYG_HOME)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ.pop('TRANSFORMERS_CACHE', None)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PROSST_HOME'] = str(PROSST_HOME)

def checkout_complete(home, required_files):
    return home.is_dir() and all((home / relative).is_file() for relative in required_files)

def add_project_paths():
    for path in [SAPROT_HOME, PROSST_HOME]:
        value = str(path)
        if value not in sys.path:
            sys.path.insert(0, value)

def runtime_environment():
    env = os.environ.copy()
    project_paths = [str(SAPROT_HOME), str(PROSST_HOME)]
    if env.get('PYTHONPATH'):
        project_paths.append(env['PYTHONPATH'])
    env['PYTHONPATH'] = os.pathsep.join(project_paths)
    return env

def probe_runtime():
    if not checkout_complete(SAPROT_HOME, SAPROT_REQUIRED):
        return False, 'SaprotHub checkout is missing ColabProSST files.'
    if not checkout_complete(PROSST_HOME, PROSST_REQUIRED):
        return False, 'Official ProSST checkout is incomplete.'

    probe = subprocess.run(
        [
            sys.executable,
            '-c',
            (
                'import importlib; import saprot; import numpy; import pandas; '
                'import torch; import torch_geometric; import torch_scatter; '
                "importlib.import_module('prosst.structure.get_sst_seq'); "
                'from saprot.utils.colab_prosst_workflow import ColabProSSTWorkflow'
            ),
        ],
        env=runtime_environment(),
        capture_output=True,
        text=True,
    )
    details = (probe.stderr or probe.stdout).strip()
    return probe.returncode == 0, details

def run_command(args):
    subprocess.run([str(arg) for arg in args], check=True)

def run_pip(*args):
    run_command([sys.executable, '-m', 'pip', *args])

def clone_saprothub():
    if SAPROT_HOME.exists():
        shutil.rmtree(SAPROT_HOME)
    run_command([
        'git', 'clone', '--depth', '1', '--branch', SAPROTHUB_BRANCH,
        SAPROTHUB_REPO, SAPROT_HOME,
    ])
    if not checkout_complete(SAPROT_HOME, SAPROT_REQUIRED):
        raise RuntimeError(
            f'{SAPROTHUB_REPO}@{SAPROTHUB_BRANCH} does not contain ColabProSST.'
        )

def ensure_official_prosst():
    if checkout_complete(PROSST_HOME, PROSST_REQUIRED):
        return
    if PROSST_HOME.exists():
        shutil.rmtree(PROSST_HOME)
    run_command([
        'git', 'clone', '--depth', '1',
        'https://github.com/ai4protein/ProSST.git', PROSST_HOME,
    ])
    if not checkout_complete(PROSST_HOME, PROSST_REQUIRED):
        raise RuntimeError('The official ProSST checkout is incomplete after cloning.')

runtime_ready, probe_details = probe_runtime()
if not runtime_ready:
    print('Installing ProSST...')
    clone_saprothub()
    ensure_official_prosst()

    # Match ColabSaprot's environment order, with only ProSST quantizer extras added.
    run_pip('install', '-r', SAPROT_HOME / 'requirements.txt')
    run_pip('install', SAPROT_HOME)
    run_pip('install', 'jupyter_ui_poll')
    run_pip('uninstall', '-y', 'torchao')
    run_pip('install', 'joblib==1.3.2', 'pathos==0.3.2', 'biotite==0.39.0')

    torch_info = subprocess.check_output(
        [
            sys.executable, '-c',
            "import torch; print(torch.__version__.split('+')[0]); print(torch.version.cuda or 'cpu')",
        ],
        text=True,
    ).strip().splitlines()
    torch_version, torch_cuda = torch_info
    if torch_version.startswith('2.2.'):
        torch_version = '2.2.0'
    pyg_backend = 'cpu' if torch_cuda == 'cpu' else 'cu' + torch_cuda.replace('.', '')
    pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{pyg_backend}.html'
    print(f'Installing ProSST PyG extensions from {pyg_wheel_url}')
    run_pip(
        'install', '--no-index', 'pyg_lib', 'torch_scatter', 'torch_sparse',
        'torch_cluster', 'torch_spline_conv', '-f', pyg_wheel_url,
    )
    run_pip('install', 'torch-geometric==2.5.0')

    for path in [
        SAPROT_HOME / 'LMDB',
        SAPROT_HOME / 'output',
        SAPROT_HOME / 'datasets',
        SAPROT_HOME / 'adapters' / 'classification' / 'Local',
        SAPROT_HOME / 'adapters' / 'regression' / 'Local',
        SAPROT_HOME / 'structures',
    ]:
        path.mkdir(parents=True, exist_ok=True)
    for binary in (SAPROT_HOME / 'bin').glob('*'):
        binary.chmod(binary.stat().st_mode | 0o111)

    # Same in-kernel NumPy refresh used by ColabSaprot after binary installs.
    for module_name in list(sys.modules):
        if module_name == 'numpy' or module_name.startswith('numpy.'):
            del sys.modules[module_name]
    importlib.invalidate_caches()

    runtime_ready, probe_details = probe_runtime()
    if not runtime_ready:
        raise RuntimeError(
            'ColabProSST installation finished, but the isolated import check failed:\n'
            + probe_details
        )

add_project_paths()
importlib.import_module('prosst.structure.get_sst_seq')
from saprot.utils.colab_prosst_workflow import ColabProSSTWorkflow

COLABPROSST_WORKFLOW = ColabProSSTWorkflow()
COLABPROSST_WORKFLOW.create_csv_templates(download=DOWNLOAD_CSV_TEMPLATES)
print('ProSST is installed successfully!')


In [ ]:
#@title **Run ColabProSST**
from IPython.display import display

WORKFLOW = 'Convert structure to tokens' #@param ['Convert structure to tokens', 'Zero-shot mutation effect prediction', 'Fine-tune classification', 'Fine-tune regression', 'Predict classification', 'Predict regression']
INPUT_CSV = '' #@param {type:'string'}
UPLOAD_INPUT_CSV = False #@param {type:'boolean'}
STRUCTURE_FILE = '' #@param {type:'string'}
UPLOAD_STRUCTURE_FILE = False #@param {type:'boolean'}
CHAIN_ID = '' #@param {type:'string'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
CHECKPOINT_PATH = '' #@param {type:'string'}
TASK_NAME = 'ProSSTUserTask' #@param {type:'string'}
NUM_LABELS = 2 #@param {type:'integer'}
MAX_EPOCHS = 2 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FREEZE_BACKBONE = False #@param {type:'boolean'}
DOWNLOAD_OUTPUTS = True #@param {type:'boolean'}

# Advanced defaults. Most users can leave these unchanged for the first ProSST-2048 version.
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
STRUCTURE_VOCAB_SIZE = 2048 #@param {type:'integer'}
GRADIENT_CHECKPOINTING = True #@param {type:'boolean'}
OUTPUT_DIR = '/content/colabprosst_outputs' #@param {type:'string'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

COLABPROSST_WORKFLOW.set_output_dir(OUTPUT_DIR)

if WORKFLOW == 'Convert structure to tokens':
    result_df = COLABPROSST_WORKFLOW.convert_structure(
        structure_path=STRUCTURE_FILE,
        upload_structure=UPLOAD_STRUCTURE_FILE,
        chain_id=CHAIN_ID,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df)

elif WORKFLOW == 'Zero-shot mutation effect prediction':
    result_df = COLABPROSST_WORKFLOW.run_zero_shot(
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())

elif WORKFLOW in ['Fine-tune classification', 'Fine-tune regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    result = COLABPROSST_WORKFLOW.train_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        task_name=TASK_NAME,
        num_labels=NUM_LABELS,
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        freeze_backbone=FREEZE_BACKBONE,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        download=DOWNLOAD_OUTPUTS,
    )
    print(result)

elif WORKFLOW in ['Predict classification', 'Predict regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    if not CHECKPOINT_PATH.strip():
        raise ValueError('Set CHECKPOINT_PATH before running prediction.')
    result_df = COLABPROSST_WORKFLOW.predict_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        checkpoint_path=CHECKPOINT_PATH,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        num_labels=NUM_LABELS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())


In [ ]:
#@title **Optional: upload a trained ColabProSST checkpoint to Hugging Face**
HF_REPO_ID = '' #@param {type:'string'}
HF_PRIVATE_REPO = False #@param {type:'boolean'}
RUN_HF_LOGIN = True #@param {type:'boolean'}
CHECKPOINT_PATH = '/content/SaprotHub/weights/prosst/ProSSTUserTask.pt' #@param {type:'string'}
TASK_TYPE = 'classification' #@param ['classification', 'regression']
NUM_LABELS = 2 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
MODEL_CARD_TITLE = 'ColabProSST model' #@param {type:'string'}
MODEL_DESCRIPTION = 'A ProSST checkpoint trained with ColabProSST.' #@param {type:'string'}
DOWNLOAD_MODEL_PACKAGE = False #@param {type:'boolean'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

package_dir = COLABPROSST_WORKFLOW.upload_checkpoint_to_hf(
    repo_id=HF_REPO_ID,
    checkpoint_path=CHECKPOINT_PATH,
    task_type=TASK_TYPE,
    num_labels=NUM_LABELS,
    model_path=MODEL_PATH,
    private=HF_PRIVATE_REPO,
    run_login=RUN_HF_LOGIN,
    title=MODEL_CARD_TITLE,
    description=MODEL_DESCRIPTION,
    download_package=DOWNLOAD_MODEL_PACKAGE,
)
print('local package:', package_dir)
